In [ ]:
# This notebook will be used to run the regression models to look at annual growth
import sqlalchemy 
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from scipy import stats
import scikit_posthocs as sp
# import statsmodels.api as sm
import statsmodels.formula.api as smf # this is to run a lm like R

In [ ]:
load_dotenv("../../.env")

mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_password = os.environ.get("MYSQL_PASSWORD")
mysql_database = os.environ.get("MYSQL_DATABASE")


# Creates the engine, a reusable blueprint for connecting to MySQL.
# This does not open a connection itself, it just stores the driver,
# credentials, host, and database name.
engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_database}")

with engine.connect() as conn:
    print("Connection successful")

In [ ]:
#I am importing the data from MySQL and creating the daily returns variable

daily_prices = pd.read_sql("SELECT ticker, date, adjusted_close FROM daily_prices ", engine)

daily_prices.head()

daily_prices['time'] = daily_prices.groupby('ticker').cumcount() + 1
daily_prices['time_C'] = daily_prices['time'] - daily_prices['time'].mean() 
daily_prices.head(10)

In [ ]:
daily_prices.groupby('ticker')['adjusted_close'].mean()

In [ ]:
# fitting a separate linear regression per company, adjusted_close ~ centered time
# slope and r_squared describe price growth rate and how linear that growth was
# p-values are intentionally not stored, see METHODOLOGY.md, daily prices are autocorrelated
# so the standard errors and p-values from this regression can't be trusted as inference
results_list = []

for ticker in daily_prices['ticker'].unique():
    temp_df = daily_prices[daily_prices['ticker'] == ticker]
    returnbytime = smf.ols("adjusted_close ~ time_C", data=temp_df)
    results = returnbytime.fit()
    results_list.append({
        'ticker': ticker,
        'regression_slope' : results.params['time_C'],
        'r_squared':  results.rsquared,
        'intercept': results.params['Intercept']
            })
    print(f'\n\n the ticker is {ticker}', results.summary())
    
results_df = pd.DataFrame(results_list)
results_df.head()

# truncate first since this notebook is rerun whenever the regression needs updating,
# without this, results would duplicate every time the cell runs
with engine.begin() as conn:
    conn.execute(sqlalchemy.text("TRUNCATE TABLE regression_results"))
results_df.to_sql(name = 'regression_results', con = engine, if_exists = 'append', index = False)

In [ ]:
# Inferential tests parked: daily stock observations are autocorrelated,
# which violates the independence assumption of Kruskal-Wallis.
# p-values are not trustworthy here, so these are not used in the analysis.
# Kept for reference and to show the work, see METHODOLOGY.md for full reasoning.
# Stability (CV of daily range)
# cv_data = pd.read_sql("SELECT ticker, cv_daily_range FROM monthly_summary", engine)
# stats.kruskal(cv_data[cv_data['ticker'] == 'DLR']['cv_daily_range'], 
#               cv_data[cv_data['ticker'] == 'EQIX']['cv_daily_range'], 
#               cv_data[cv_data['ticker'] == 'IRM']['cv_daily_range'])
# sp.posthoc_dunn(cv_data, 'cv_daily_range', 'ticker', p_adjust='bonferroni')
# cv_data.groupby('ticker')['cv_daily_range'].median()

# Growth (daily returns)
# stats.kruskal(daily_prices[daily_prices['ticker'] == 'DLR']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'EQIX']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'IRM']['daily_returns'].dropna())